# Gradient Boosting for Opener prediction model

In [1]:
import pandas as pd

data = pd.read_csv("./Training_Data.csv")
data.head()

,name,position,number_of_maps,firepower,entrying,trading,opening,clutching,sniping,utility,igl
0,alex666,Closer,83,27,44,43,26,58,0,61,1
1,npl,Opener,83,85,44,60,76,47,0,48,0
2,kensizor,Opener,83,70,94,56,63,37,0,41,0
3,esenthial,Linchpin,83,31,40,79,22,61,0,23,0
4,s1zzi,AWPer,83,57,47,32,77,63,93,33,0


In [2]:
feature_cols = [
    "firepower",
    "entrying",
    "trading",
    "opening",
    "clutching",
    "sniping",
    "utility",
    "igl",
]

x = data[feature_cols]
y = data["position"]

y.replace("Closer", 0, inplace=True)
y.replace("Linchpin", 0, inplace=True)
y.replace("Opener", 1, inplace=True)
y.replace("AWPer", 0, inplace=True)

y.head()

/tmp/ipykernel_6818/1539686208.py:18: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  y.replace("AWPer", 0, inplace=True)


0    0
1    1
2    1
3    0
4    0
Name: position, dtype: int64

In [3]:
import numpy as np
from xgboost import XGBClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    ConfusionMatrixDisplay,
)
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from scipy.stats import randint, uniform


x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y
)

In [4]:
param_dist = {
    "n_estimators": randint(100, 600),
    "max_depth": randint(3, 8),
    "learning_rate": uniform(loc=0.01, scale=0.3),
    "subsample": uniform(0.7, 0.3),
    "colsample_bytree": uniform(0.7, 0.3),
}


gb = XGBClassifier(random_state=42, n_jobs=1, objective="binary:logistic")

rand_search = RandomizedSearchCV(
    gb,
    param_distributions=param_dist,
    n_iter=5,
    cv=5,
    scoring="accuracy",
    n_jobs=2,
    random_state=42,
).fit(x_train, y_train)

best_rf = rand_search.best_estimator_

params = rand_search.best_params_

print("Best hyperparameters:", params)

Best hyperparameters: {'colsample_bytree': np.float64(0.7468055921327309), 'learning_rate': np.float64(0.05679835610086079), 'max_depth': 5, 'n_estimators': 558, 'subsample': np.float64(0.9598528437324805)}


In [5]:
model = XGBClassifier(**params, objective="binary:logistic")
model.fit(x_train, y_train)

y_pred = model.predict(x_test)

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy: ", accuracy)

Accuracy:  0.8181818181818182


In [6]:
print(y_test)
print(y_pred)

141    0
3      0
96     0
156    0
148    0
18     1
115    0
107    1
102    0
106    1
174    1
85     0
84     0
162    0
12     1
4      0
154    1
170    0
87     1
133    0
35     0
125    1
122    0
120    0
163    0
60     0
47     0
166    0
172    0
17     1
32     0
39     0
68     1
95     0
28     1
159    0
155    1
52     0
101    1
140    0
58     1
83     1
22     1
24     0
Name: position, dtype: int64
[0 0 0 0 0 1 0 1 0 1 1 0 0 1 1 0 1 0 0 0 0 1 1 1 0 0 1 0 1 0 1 0 1 0 1 0 1
 0 1 0 1 1 1 0]


In [7]:
print(x_test)

     firepower  entrying  trading  opening  clutching  sniping  utility  igl
141         67         6       70       75         40       86       66    0
3           31        40       79       22         61        0       23    0
96          50        13       21       38         71        0       24    0
156         23        61       21       27         52        0       67    0
148         19        15       42       42         84        0       66    0
18          93        83       47       88         48        0       71    0
115         70         6       43       48         50        3       21    0
107         90        43       55       88         30        0       79    0
102         29        65       29       27         46        0       73    0
106         38        76       21       83         18        3       75    1
174         77        45       31       70         21        2       37    0
85          71         6       22       84         62        2       24    0

In [14]:
p = [84, 45, 55, 75, 39, 94, 28, 0]
player = pd.DataFrame([p])
player.columns = feature_cols
print(model.predict(player))

[1]


In [15]:
print(model.predict_proba(player))

[[0.35333872 0.6466613 ]]


In [16]:
model.save_model("opener.json")